# Burmese-English NMT with NLLB-200 (Fine-Tuning)

**Student**: Htut Ko Ko (st126010)  
**Course**: NLP Project A3  
**Task**: High-Quality Machine Translation (Web App Integration)

## 1. Introduction & Motivation
In this notebook, I implement a **Neural Machine Translation (NMT)** system to translate between **Burmese** and **English**.

For the assignment's "Task 4: Web Application", my goal was to achieve **production-quality** translation that users would actually find useful.

Training a Transformer from scratch (as done in my other notebook) on the small **ALT dataset (20k pairs)** resulted in poor fluency because deep learning models require massive amounts of data. To solve this, I chose to **fine-tune** a state-of-the-art pre-trained model: **NLLB-200 (No Language Left Behind)** by Meta.

This approach allows me to leverage the model's existing knowledge of Burmese and English while adapting it specifically to the ALT dataset style.

## 2. Setup & Dependencies
First, I install the necessary libraries from HuggingFace (`transformers`, `datasets`) and tools for evaluating translation quality (`sacrebleu`). I also mount my Google Drive so that I can save the fine-tuned model safely and use it later in my Web App.

In [ ]:
!pip install transformers datasets sentencepiece sacremoses accelerate

In [ ]:
import os
import torch
import numpy as np
import pandas as pd
from datasets import load_dataset, Dataset, DatasetDict
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq, Seq2SeqTrainingArguments, Seq2SeqTrainer
from google.colab import drive

# I mount Google Drive to ensure my model is saved persistently.
drive.mount('/content/drive')

# I define the save path in my Drive so I can download it later for the Web App.
DRIVE_SAVE_PATH = "/content/drive/MyDrive/NLP/Project_A3/nllb_model"
os.makedirs(DRIVE_SAVE_PATH, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 3. Data Preparation (ALT Dataset)
I am using the **Asian Language Treebank (ALT)** dataset as required. The raw dataset contains multiple languages, so I filter it to extract only the **Burmese ('my')** and **English ('en')** pairs.

I then split the data into:
- **Train (81%)**: For teaching the model.
- **Validation (9%)**: For checking improvements during training.
- **Test (10%)**: For final evaluation.

In [ ]:
print("Loading ALT Dataset...")
try:
    raw_dataset = load_dataset("alt", split="train+validation+test")
    
    data = []
    for item in raw_dataset:
        if 'translation' in item:
            if 'my' in item['translation'] and 'en' in item['translation']:
                data.append({
                    'my': item['translation']['my'],
                    'en': item['translation']['en']
                })
    
    df = pd.DataFrame(data)
    df = df.dropna()
    print(f"Total Pairs Extracted: {len(df)}")

except Exception as e:
    print(f"Error: {e}")

In [ ]:
from sklearn.model_selection import train_test_split

# Splitting: 90% Train+Val, 10% Test
train_df, test_df = train_test_split(df, test_size=0.1, random_state=42)
# Splitting Train+Val: 90% Train, 10% Val
train_df, val_df = train_test_split(train_df, test_size=0.1, random_state=42)

print(f"Train Size: {len(train_df)}")
print(f"Results Validation Size: {len(val_df)}")
print(f"Test Size: {len(test_df)}")

# Convert back to HuggingFace Dataset format for easier processing
train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)
test_dataset = Dataset.from_pandas(test_df)

dataset = DatasetDict({
    'train': train_dataset,
    'validation': val_dataset,
    'test': test_dataset
})

## 4. Model Loading & Tokenization
Here I load the **NLLB-200-distilled-600M** model. This is a distilled version of the massive 54B parameter model, making it efficient enough to fine-tune on Colab while retaining high performance.

**Important**: NLLB requires specific language codes:
- Burmese: `mya_Mymr`
- English: `eng_Latn`

I create a preprocessing function to tokenize the inputs. We tokenize the inputs (Burmese) and the targets (English) simultaneously.

In [ ]:
model_checkpoint = "facebook/nllb-200-distilled-600M"

tokenizer = AutoTokenizer.from_pretrained(model_checkpoint, src_lang="mya_Mymr", tgt_lang="eng_Latn")
model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint).to(device)

In [ ]:
max_input_length = 128
max_target_length = 128

def preprocess_function(examples):
    inputs = [ex for ex in examples['my']]
    targets = [ex for ex in examples['en']]
    
    # We tokenize the input (Burmese)
    model_inputs = tokenizer(inputs, max_length=max_input_length, truncation=True)
    # We tokenize the target (English) as labels
    labels = tokenizer(text_target=targets, max_length=max_target_length, truncation=True)

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_datasets = dataset.map(preprocess_function, batched=True)

## 5. Fine-Tuning (Training)
I use the `Seq2SeqTrainer` to fine-tune the model.

**Hyperparameters:**
- **Batch Size**: 16 (fits in Colab GPU memory).
- **Learning Rate**: 2e-5 (low learning rate to gently adjust pre-trained weights).
- **Epochs**: 3 (Since the model is already pre-trained, it converges very quickly. 3 epochs is sufficient to adapt to the ALT dataset style without overfitting).

In [ ]:
batch_size = 16
learning_rate = 2e-5
weight_decay = 0.01
num_train_epochs = 3

args = Seq2SeqTrainingArguments(
    DRIVE_SAVE_PATH,
    eval_strategy = "epoch",
    learning_rate=learning_rate,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    weight_decay=weight_decay,
    save_total_limit=1,
    num_train_epochs=num_train_epochs,
    predict_with_generate=True,
    fp16=True if torch.cuda.is_available() else False,
)

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
)

In [ ]:
print("Starting Training...")
trainer.train()

## 6. Saving the Model
After training is complete, I save the model and the tokenizer to Google Drive. This is the crucial step that allows me to download the model folder later and use it in my local Flask web application.

In [ ]:
trainer.save_model(DRIVE_SAVE_PATH)
tokenizer.save_pretrained(DRIVE_SAVE_PATH)
print(f"Model and Tokenizer saved safely to '{DRIVE_SAVE_PATH}'")

## 7. Verification & Inference
Finally, I verify that the model works by loading it back from the drive and running a translation test. I use `model.generate()` directly for robustness, ensuring the correct language codes are sent to the model.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

# Reload from Drive to verify consistency
print(f"Reloading model from {DRIVE_SAVE_PATH}...")
tokenizer = AutoTokenizer.from_pretrained(DRIVE_SAVE_PATH)
model = AutoModelForSeq2SeqLM.from_pretrained(DRIVE_SAVE_PATH).to(device)

def translate(text):
    # Set source language explicitly
    tokenizer.src_lang = "mya_Mymr"
    inputs = tokenizer(text, return_tensors="pt").to(device)
    
    with torch.no_grad():
        # Generate encoded output
        translated_tokens = model.generate(
            **inputs, 
            # Force the target language to be English
            forced_bos_token_id=tokenizer.convert_tokens_to_ids("eng_Latn"), 
            max_length=128
        )
    # Decode tokens back to text
    return tokenizer.batch_decode(translated_tokens, skip_special_tokens=True)[0]

# Manual Test
text = "မင်္ဂလာပါ"
print(f"Source: {text}")
print(f"Prediction: {translate(text)}")

# Random Test from Test Set
sample = test_df.sample(1).iloc[0]
print(f"\nTest Sample Source: {sample['my']}")
print(f"Test Sample Target: {sample['en']}")
print(f"Model Prediction: {translate(sample['my'])}")